# Caso 2: Agente avanzado de ventas y restock de inventario

Objetivo: pronosticar demanda, calcular riesgo de inventario, recomendar restock y generar un dashboard ejecutivo avanzado con agente IA dinámico.

## Qué produce este notebook

- Pronóstico de demanda a 30 días por producto.
- Cálculo de stock de seguridad, punto de reorden y restock sugerido.
- Métricas de desempeño del modelo.
- Dashboard HTML interactivo estilo magIA/Bloomberg.
- Agente IA renderizado como chat con animación, preguntas rápidas y recomendaciones accionables.

In [ ]:
# Instalación mínima para Colab
# Si ya están instaladas, esta celda no causa problema.
!pip -q install pandas numpy scikit-learn plotly openpyxl yfinance

import os
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from IPython.display import HTML, display, IFrame

# Opcional: montar Google Drive en Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Corporate_AI_Lab'
    os.makedirs(BASE_DIR, exist_ok=True)
except Exception:
    BASE_DIR = '.'

print('Carpeta de trabajo:', BASE_DIR)

def calcular_rmse(y_real, y_pred):
    """Calcula RMSE sin depender de mean_squared_error(..., squared=False)."""
    return np.sqrt(mean_squared_error(y_real, y_pred))


In [ ]:
# Cargar base: intenta Drive; si no existe, usa archivo local cargado al entorno
possible_paths = [
    os.path.join(BASE_DIR, 'ventas_inventario_sintetico.csv'),
    '/content/ventas_inventario_sintetico.csv',
    'ventas_inventario_sintetico.csv'
]
path = next((p for p in possible_paths if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError('Sube ventas_inventario_sintetico.csv o guárdalo en Corporate_AI_Lab dentro de Drive')

df = pd.read_csv(path)
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.sort_values(['producto','fecha']).reset_index(drop=True)
print(df.shape)
df.head()


In [ ]:
# Ingeniería de variables para pronóstico de demanda
# Se construyen variables de calendario, rezagos y promedios móviles por producto.

df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['semana'] = df['fecha'].dt.isocalendar().week.astype(int)
df['dia_mes'] = df['fecha'].dt.day
df['ingreso'] = df['ventas_unidades'] * df['precio']

for lag in [1, 7, 14]:
    df[f'ventas_lag_{lag}'] = df.groupby('producto')['ventas_unidades'].shift(lag)

for win in [7, 14, 30]:
    df[f'ventas_ma_{win}'] = (
        df.groupby('producto')['ventas_unidades']
        .shift(1)
        .rolling(win)
        .mean()
        .reset_index(level=0, drop=True)
    )

# Variable de presión de inventario
# Valores altos indican que las ventas recientes están consumiendo inventario con rapidez.
df['ratio_ventas_inventario'] = df['ventas_unidades'] / df['inventario_inicial'].replace(0, np.nan)
df['ratio_ventas_inventario'] = df['ratio_ventas_inventario'].fillna(0)

df_model = df.dropna().copy()

base_features = [
    'dia_semana','mes','semana','dia_mes','promocion','precio','inventario_inicial',
    'lead_time_dias','ventas_lag_1','ventas_lag_7','ventas_lag_14',
    'ventas_ma_7','ventas_ma_14','ventas_ma_30','ratio_ventas_inventario'
]
cat_features = ['producto','region','canal']

df_encoded = pd.get_dummies(df_model[base_features + cat_features], columns=cat_features, drop_first=False)
features = df_encoded.columns.tolist()
X = df_encoded
y = df_model['ventas_unidades']

# División temporal: últimos 20% como prueba, sin mezclar aleatoriamente el tiempo
split_idx = int(len(df_model) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = RandomForestRegressor(n_estimators=350, random_state=42, min_samples_leaf=2, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = calcular_rmse(y_test, pred)
r2 = r2_score(y_test, pred)
print(f'MAE: {mae:,.2f} unidades')
print(f'RMSE: {rmse:,.2f} unidades')
print(f'R2: {r2:.3f}')

backtest = df_model.iloc[split_idx:].copy()
backtest['prediccion'] = pred
backtest[['fecha','producto','ventas_unidades','prediccion']].head()


In [ ]:
# Pronóstico a 30 días por producto
# Para que la actividad sea estable y didáctica, el pronóstico usa supuestos controlados sobre precio/promoción/inventario.

future_rows = []
productos = sorted(df['producto'].unique())
last_date = df['fecha'].max()

# Datos auxiliares para columnas dummy consistentes
train_columns = features

for producto in productos:
    hist_prod = df[df['producto'] == producto].sort_values('fecha').copy()
    region_mode = hist_prod['region'].mode().iloc[0]
    canal_mode = hist_prod['canal'].mode().iloc[0]
    lead_time = int(hist_prod['lead_time_dias'].median())
    precio_base = float(hist_prod['precio'].tail(30).mean())
    inventario_actual = float(hist_prod['inventario_final'].iloc[-1])
    temp = hist_prod.copy()
    for h in range(1, 31):
        fdate = last_date + pd.Timedelta(days=h)
        promo = 1 if fdate.dayofweek in [4,5] and h % 3 == 0 else 0
        inv_ini = max(0, inventario_actual)
        row = {
            'fecha': fdate,
            'producto': producto,
            'region': region_mode,
            'canal': canal_mode,
            'promocion': promo,
            'precio': precio_base * (0.985 if promo else 1.0),
            'inventario_inicial': inv_ini,
            'lead_time_dias': lead_time,
            'dia_semana': fdate.dayofweek,
            'mes': fdate.month,
            'semana': int(fdate.isocalendar().week),
            'dia_mes': fdate.day,
            'ventas_lag_1': temp['ventas_unidades'].iloc[-1],
            'ventas_lag_7': temp['ventas_unidades'].iloc[-7] if len(temp) >= 7 else temp['ventas_unidades'].mean(),
            'ventas_lag_14': temp['ventas_unidades'].iloc[-14] if len(temp) >= 14 else temp['ventas_unidades'].mean(),
            'ventas_ma_7': temp['ventas_unidades'].tail(7).mean(),
            'ventas_ma_14': temp['ventas_unidades'].tail(14).mean(),
            'ventas_ma_30': temp['ventas_unidades'].tail(30).mean(),
            'ratio_ventas_inventario': temp['ventas_unidades'].tail(7).mean() / max(inv_ini, 1)
        }
        row_df = pd.DataFrame([row])
        row_encoded = pd.get_dummies(row_df[base_features + cat_features], columns=cat_features, drop_first=False)
        row_encoded = row_encoded.reindex(columns=train_columns, fill_value=0)
        demand_pred = max(0, float(model.predict(row_encoded)[0]))
        row['demanda_pronosticada'] = demand_pred
        future_rows.append(row)
        # actualiza inventario sintético para el siguiente día
        inventario_actual = max(0, inventario_actual - demand_pred)
        temp = pd.concat([temp, pd.DataFrame([{**row, 'ventas_unidades': demand_pred, 'inventario_final': inventario_actual}])], ignore_index=True)

forecast = pd.DataFrame(future_rows)

# Resumen ejecutivo por producto
summary = []
for producto in productos:
    hist_prod = df[df['producto'] == producto].sort_values('fecha')
    fut_prod = forecast[forecast['producto'] == producto]
    inv_actual = float(hist_prod['inventario_final'].iloc[-1])
    lead_time = int(hist_prod['lead_time_dias'].median())
    demanda_diaria_prom = float(fut_prod['demanda_pronosticada'].mean())
    demanda_30d = float(fut_prod['demanda_pronosticada'].sum())
    sigma_diaria = float(hist_prod['ventas_unidades'].tail(60).std())
    stock_seguridad = 1.65 * sigma_diaria * np.sqrt(lead_time)
    demanda_lt = demanda_diaria_prom * lead_time
    punto_reorden = demanda_lt + stock_seguridad
    restock_sugerido = max(0, demanda_30d + stock_seguridad - inv_actual)
    dias_cobertura = inv_actual / max(demanda_diaria_prom, 1)
    if inv_actual < punto_reorden:
        riesgo = 'ALTO'
    elif inv_actual < punto_reorden * 1.25:
        riesgo = 'MEDIO'
    else:
        riesgo = 'BAJO'
    summary.append({
        'producto': producto,
        'inventario_actual': inv_actual,
        'demanda_30d': demanda_30d,
        'demanda_diaria_prom': demanda_diaria_prom,
        'lead_time_dias': lead_time,
        'stock_seguridad': stock_seguridad,
        'punto_reorden': punto_reorden,
        'restock_sugerido': restock_sugerido,
        'dias_cobertura': dias_cobertura,
        'riesgo_stockout': riesgo
    })

restock = pd.DataFrame(summary).sort_values(['riesgo_stockout','restock_sugerido'], ascending=[True, False])
restock


In [ ]:
# Agente ejecutivo dinámico basado en reglas
alto = int((restock['riesgo_stockout'] == 'ALTO').sum())
medio = int((restock['riesgo_stockout'] == 'MEDIO').sum())
producto_critico = restock.sort_values('restock_sugerido', ascending=False).iloc[0]['producto']
restock_total = float(restock['restock_sugerido'].sum())
demanda_total = float(restock['demanda_30d'].sum())

if alto > 0:
    nivel = 'ALTO'
    accion = f'priorizar compra y reposición inmediata del producto {producto_critico}; validar proveedor alterno y confirmar lead time real.'
elif medio > 0:
    nivel = 'MEDIO'
    accion = 'monitorear diariamente inventario, activar órdenes parciales y revisar promociones que aceleren la salida de stock.'
else:
    nivel = 'BAJO'
    accion = 'mantener política actual, revisar oportunidades de optimización de capital de trabajo y evitar sobreinventario.'

agent_messages = [
    {
        'role': 'assistant',
        'title': 'Lectura ejecutiva del agente',
        'text': f'El modelo estima una demanda total de {demanda_total:,.0f} unidades para los próximos 30 días. El nivel consolidado de riesgo de inventario es {nivel}.'
    },
    {
        'role': 'assistant',
        'title': 'Producto prioritario',
        'text': f'El producto que requiere mayor atención es {producto_critico}. La recomendación inicial es asignar {restock_total:,.0f} unidades de restock agregado, ajustando por lead time y stock de seguridad.'
    },
    {
        'role': 'assistant',
        'title': 'Acción sugerida',
        'text': f'Se recomienda {accion}'
    },
    {
        'role': 'assistant',
        'title': 'Nota metodológica',
        'text': f'El modelo Random Forest usa calendario, precio, promoción, inventario, lead time, rezagos de demanda y promedios móviles. Validación: MAE {mae:,.1f}, RMSE {rmse:,.1f}, R² {r2:.3f}.'
    }
]

quick_answers = {
    '¿Qué producto priorizo?': f'Prioriza {producto_critico}, porque concentra la mayor necesidad estimada de restock y puede detonar quiebre de stock si el lead time se mantiene.',
    '¿Qué acción tomaría hoy?': f'Generaría una orden de compra parcial por las unidades sugeridas para productos con riesgo ALTO, comenzando por {producto_critico}.',
    '¿Qué debo monitorear?': 'Monitorea demanda diaria real, inventario final, cumplimiento de proveedor, promociones activas y desviación entre demanda real y pronosticada.'
}

print('Nivel de riesgo:', nivel)
print('Restock total sugerido:', f'{restock_total:,.0f}')


In [ ]:
# Dashboard HTML avanzado: ventas + restock + agente IA dinámico
import os, json

# -----------------------------
# Visualizaciones Plotly
# -----------------------------
colors = {
    'bg':'#07111f','card':'#0d1d36','cyan':'#22d8ff','green':'#42f5b3','amber':'#ffbf69','red':'#ff4d6d','purple':'#9b5cff','text':'#eaf6ff'
}

def layout_fig(fig, title=None, height=430):
    fig.update_layout(
        template='plotly_dark',
        height=height,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=40,r=20,t=60,b=40),
        font=dict(color=colors['text'], family='Inter, Arial'),
        title=dict(text=title or '', x=0.02, xanchor='left', font=dict(size=18)),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.update_xaxes(gridcolor='rgba(255,255,255,.08)')
    fig.update_yaxes(gridcolor='rgba(255,255,255,.08)')
    return fig

hist_90 = df[df['fecha'] >= df['fecha'].max() - pd.Timedelta(days=90)].copy()
fig_hist = px.line(hist_90, x='fecha', y='ventas_unidades', color='producto', markers=False)
layout_fig(fig_hist, 'Demanda histórica reciente por producto')

fig_forecast = px.line(forecast, x='fecha', y='demanda_pronosticada', color='producto', markers=True)
layout_fig(fig_forecast, 'Pronóstico de demanda a 30 días')

fig_restock = go.Figure()
fig_restock.add_bar(x=restock['producto'], y=restock['inventario_actual'], name='Inventario actual', marker_color=colors['cyan'])
fig_restock.add_bar(x=restock['producto'], y=restock['punto_reorden'], name='Punto de reorden', marker_color=colors['amber'])
fig_restock.add_bar(x=restock['producto'], y=restock['restock_sugerido'], name='Restock sugerido', marker_color=colors['green'])
fig_restock.update_layout(barmode='group')
layout_fig(fig_restock, 'Inventario vs punto de reorden y restock')

risk_color_map = {'ALTO':'#ff4d6d','MEDIO':'#ffbf69','BAJO':'#42f5b3'}
fig_risk = px.treemap(restock, path=['riesgo_stockout','producto'], values='demanda_30d', color='riesgo_stockout', color_discrete_map=risk_color_map)
layout_fig(fig_risk, 'Mapa de riesgo por producto', height=430)

backtest_sample = backtest.sort_values('fecha').tail(250)
fig_backtest = px.scatter(backtest_sample, x='ventas_unidades', y='prediccion', color='producto', trendline='ols')
fig_backtest.add_trace(go.Scatter(x=[backtest_sample['ventas_unidades'].min(), backtest_sample['ventas_unidades'].max()], y=[backtest_sample['ventas_unidades'].min(), backtest_sample['ventas_unidades'].max()], mode='lines', name='Línea perfecta', line=dict(dash='dash', color='white')))
layout_fig(fig_backtest, 'Backtest: demanda real vs predicha')

fi = pd.DataFrame({'variable': features, 'importancia': model.feature_importances_}).sort_values('importancia', ascending=False).head(15)
fig_fi = px.bar(fi.sort_values('importancia'), x='importancia', y='variable', orientation='h')
layout_fig(fig_fi, 'Variables más importantes del modelo')

figs = {
    'hist': fig_hist.to_html(full_html=False, include_plotlyjs='cdn'),
    'forecast': fig_forecast.to_html(full_html=False, include_plotlyjs=False),
    'restock': fig_restock.to_html(full_html=False, include_plotlyjs=False),
    'risk': fig_risk.to_html(full_html=False, include_plotlyjs=False),
    'backtest': fig_backtest.to_html(full_html=False, include_plotlyjs=False),
    'fi': fig_fi.to_html(full_html=False, include_plotlyjs=False),
}

# Tablas ejecutivas
display_table = restock.copy()
for c in ['inventario_actual','demanda_30d','demanda_diaria_prom','stock_seguridad','punto_reorden','restock_sugerido','dias_cobertura']:
    display_table[c] = display_table[c].map(lambda x: f'{x:,.1f}')
restock_html = display_table.to_html(index=False, classes='table', border=0)

kpi_html = f"""
<div class='kpi'><span>Demanda 30 días</span><strong>{demanda_total:,.0f}</strong><small>unidades estimadas</small></div>
<div class='kpi'><span>Restock sugerido</span><strong>{restock_total:,.0f}</strong><small>unidades agregadas</small></div>
<div class='kpi'><span>Productos en riesgo alto</span><strong>{alto}</strong><small>de {len(productos)} productos</small></div>
<div class='kpi'><span>R² del modelo</span><strong>{r2:.3f}</strong><small>validación temporal</small></div>
"""

agent_json = json.dumps(agent_messages, ensure_ascii=False)
quick_json = json.dumps(quick_answers, ensure_ascii=False)

html = f"""
<!DOCTYPE html>
<html lang='es'>
<head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1.0'>
<title>magIA | Agente de Ventas y Restock</title>
<link rel='preconnect' href='https://fonts.googleapis.com'>
<link rel='preconnect' href='https://fonts.gstatic.com' crossorigin>
<link href='https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap' rel='stylesheet'>
<style>
:root {{ --bg:#050b14; --panel:#0b1628; --panel2:#0f2039; --line:rgba(34,216,255,.18); --text:#eaf6ff; --muted:#9db6cf; --cyan:#22d8ff; --green:#42f5b3; --amber:#ffbf69; --red:#ff4d6d; --purple:#9b5cff; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; font-family:Inter,Arial,sans-serif; background:radial-gradient(circle at top left, rgba(34,216,255,.14), transparent 32%), radial-gradient(circle at top right, rgba(155,92,255,.12), transparent 30%), linear-gradient(180deg,#050b14,#07111f 45%,#08182b); color:var(--text); }}
.header {{ padding:28px 34px; border-bottom:1px solid var(--line); position:sticky; top:0; z-index:10; backdrop-filter:blur(18px); background:rgba(5,11,20,.76); }}
.brand {{ display:flex; justify-content:space-between; align-items:center; gap:18px; }}
.logo {{ font-size:26px; font-weight:800; letter-spacing:-.04em; }} .logo span {{ color:var(--cyan); }}
.badge {{ padding:10px 14px; border:1px solid var(--line); border-radius:999px; color:var(--cyan); background:rgba(34,216,255,.08); font-weight:700; }}
.hero {{ padding:34px; display:grid; grid-template-columns:1.1fr .9fr; gap:20px; }}
.card {{ background:linear-gradient(180deg,rgba(15,32,57,.94),rgba(9,18,33,.94)); border:1px solid var(--line); border-radius:24px; box-shadow:0 18px 50px rgba(0,0,0,.35); overflow:hidden; }}
.hero-card {{ padding:30px; position:relative; }}
h1 {{ font-size:clamp(2.2rem,4vw,4.7rem); line-height:.98; margin:14px 0; letter-spacing:-.06em; }}
p {{ color:var(--muted); line-height:1.65; }}
.kpis {{ display:grid; grid-template-columns:repeat(4,1fr); gap:14px; padding:0 34px 22px; }}
.kpi {{ padding:18px; border:1px solid rgba(255,255,255,.07); border-radius:20px; background:rgba(255,255,255,.035); }}
.kpi span {{ color:var(--muted); display:block; font-size:.88rem; }} .kpi strong {{ display:block; font-size:1.7rem; margin:7px 0; }} .kpi small {{ color:var(--cyan); }}
.tabs {{ display:flex; gap:10px; flex-wrap:wrap; padding:10px 34px 24px; }}
.tab {{ cursor:pointer; border:1px solid var(--line); background:rgba(255,255,255,.03); color:var(--muted); padding:12px 16px; border-radius:999px; font-weight:700; }}
.tab.active {{ background:rgba(34,216,255,.14); color:#fff; border-color:rgba(34,216,255,.45); }}
.section {{ display:none; padding:0 34px 34px; }} .section.active {{ display:block; }}
.grid2 {{ display:grid; grid-template-columns:1fr 1fr; gap:18px; }}
.grid3 {{ display:grid; grid-template-columns:1.1fr .9fr; gap:18px; }}
.plot {{ padding:14px; }}
.table-wrap {{ overflow:auto; max-height:520px; }}
table.table {{ width:100%; border-collapse:collapse; min-width:900px; }} .table th,.table td {{ padding:12px 14px; border-bottom:1px solid rgba(255,255,255,.07); text-align:left; font-size:.92rem; }} .table th {{ color:#8cecff; background:rgba(34,216,255,.06); position:sticky; top:0; }}
.agent {{ display:flex; flex-direction:column; height:620px; }}
.agent-head {{ padding:18px; border-bottom:1px solid var(--line); display:flex; align-items:center; gap:12px; }}
.orb {{ width:42px; height:42px; border-radius:50%; background:radial-gradient(circle,#fff, var(--cyan) 35%, var(--purple)); box-shadow:0 0 28px rgba(34,216,255,.55); animation:pulse 2.2s infinite; }}
@keyframes pulse {{ 0%,100%{{transform:scale(1)}} 50%{{transform:scale(1.08)}} }}
.agent-body {{ padding:18px; flex:1; overflow:auto; display:flex; flex-direction:column; gap:14px; }}
.msg {{ max-width:86%; padding:14px 16px; border-radius:18px; border:1px solid rgba(255,255,255,.08); background:rgba(255,255,255,.05); opacity:0; transform:translateY(8px); animation:appear .5s forwards; }}
.msg strong {{ color:#fff; display:block; margin-bottom:6px; }} .msg p {{ margin:0; color:#cfe4ff; }}
@keyframes appear {{ to{{opacity:1; transform:translateY(0)}} }}
.typing {{ display:flex; gap:6px; padding:12px 16px; width:max-content; border-radius:18px; background:rgba(34,216,255,.10); }} .typing i {{ width:7px; height:7px; border-radius:50%; background:var(--cyan); animation:bounce 1s infinite; }} .typing i:nth-child(2){{animation-delay:.15s}} .typing i:nth-child(3){{animation-delay:.3s}} @keyframes bounce {{ 0%,80%,100%{{opacity:.3}} 40%{{opacity:1}} }}
.quick {{ display:flex; gap:8px; flex-wrap:wrap; padding:0 18px 18px; }} .quick button {{ border:1px solid var(--line); border-radius:999px; background:rgba(34,216,255,.08); color:#eaf6ff; padding:10px 12px; cursor:pointer; font-weight:700; }}
.footer {{ padding:24px 34px; color:var(--muted); border-top:1px solid var(--line); }}
@media(max-width:1000px) {{ .hero,.grid2,.grid3,.kpis{{grid-template-columns:1fr}} h1{{font-size:2.6rem}} }}
</style>
</head>
<body>
<header class='header'><div class='brand'><div class='logo'>mag<span>IA</span> · Agente de Ventas y Restock</div><div class='badge'>Python · ML · Supply Chain · Dashboard ejecutivo</div></div></header>
<section class='hero'>
  <div class='card hero-card'><div class='badge'>Modelo predictivo + agente IA</div><h1>Centro inteligente de demanda e inventario</h1><p>Pronóstico de demanda a 30 días, cálculo de punto de reorden, stock de seguridad y recomendación de restock con visualización ejecutiva.</p></div>
  <div class='card hero-card'><h2>Diagnóstico</h2><p><strong>Riesgo consolidado:</strong> {nivel}</p><p><strong>Producto prioritario:</strong> {producto_critico}</p><p><strong>Acción:</strong> {accion}</p></div>
</section>
<section class='kpis'>{kpi_html}</section>
<nav class='tabs'>
<button class='tab active' data-tab='overview'>Resumen</button><button class='tab' data-tab='forecast'>Forecast</button><button class='tab' data-tab='risk'>Riesgo y restock</button><button class='tab' data-tab='model'>Modelo</button><button class='tab' data-tab='agent'>Agente IA</button>
</nav>
<section id='overview' class='section active'><div class='grid2'><div class='card plot'>{figs['hist']}</div><div class='card plot'>{figs['forecast']}</div></div></section>
<section id='forecast' class='section'><div class='grid2'><div class='card plot'>{figs['forecast']}</div><div class='card plot'>{figs['restock']}</div></div></section>
<section id='risk' class='section'><div class='grid3'><div class='card plot'>{figs['risk']}</div><div class='card table-wrap'>{restock_html}</div></div></section>
<section id='model' class='section'><div class='grid2'><div class='card plot'>{figs['backtest']}</div><div class='card plot'>{figs['fi']}</div></div></section>
<section id='agent' class='section'><div class='grid3'><div class='card agent'><div class='agent-head'><div class='orb'></div><div><strong>Agente IA de Supply Chain</strong><p style='margin:4px 0 0'>Analiza el forecast, el riesgo y propone acciones.</p></div></div><div class='agent-body' id='agentBody'><div class='typing' id='typing'><i></i><i></i><i></i></div></div><div class='quick' id='quick'></div></div><div class='card plot'>{figs['restock']}</div></div></section>
<footer class='footer'>Dashboard generado automáticamente con Python, scikit-learn y Plotly. Diseño magIA para comunicación ejecutiva.</footer>
<script>
const tabs=document.querySelectorAll('.tab'); const sections=document.querySelectorAll('.section');
tabs.forEach(t=>t.onclick=()=>{{tabs.forEach(x=>x.classList.remove('active'));sections.forEach(s=>s.classList.remove('active'));t.classList.add('active');document.getElementById(t.dataset.tab).classList.add('active');}});
const messages={agent_json}; const quick={quick_json}; const body=document.getElementById('agentBody'); const typing=document.getElementById('typing');
function addMsg(title,text,delay=0,cls='assistant'){{setTimeout(()=>{{typing.style.display='none'; const div=document.createElement('div'); div.className='msg'; div.innerHTML=`<strong>${{title}}</strong><p>${{text}}</p>`; body.appendChild(div); body.scrollTop=body.scrollHeight;}},delay);}}
messages.forEach((m,i)=>{{setTimeout(()=>{{typing.style.display='flex';}}, i*1400); addMsg(m.title,m.text,i*1400+650);}});
const q=document.getElementById('quick'); Object.keys(quick).forEach(k=>{{const b=document.createElement('button'); b.textContent=k; b.onclick=()=>{{const user=document.createElement('div'); user.className='msg'; user.style.marginLeft='auto'; user.style.background='rgba(155,92,255,.16)'; user.innerHTML=`<strong>Pregunta ejecutiva</strong><p>${{k}}</p>`; body.appendChild(user); typing.style.display='flex'; body.appendChild(typing); addMsg('Respuesta del agente',quick[k],650);}}; q.appendChild(b);}});
</script>
</body>
</html>
"""

out_path = os.path.join(BASE_DIR, 'dashboard_ventas_restock_AVANZADO.html')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html)
print('Dashboard guardado en:', out_path)
display(HTML(f"<a href='{out_path}' target='_blank'>Abrir dashboard avanzado</a>"))
try:
    display(IFrame(out_path, width='100%', height=780))
except Exception:
    pass
